[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/skyrmions/skyrmions.ipynb)

# Magnetic Skyrmions

This example computes the crystallization, field-driven collapse, and current-driven Hall drift of magnetic skyrmions in a thin chiral magnet. Because a chiral crystal lacks mirror symmetry, neighbouring magnetic moments favour a slight relative twist of fixed handedness alongside ferromagnetic alignment. At zero external field that competition settles into a helical spiral, whereas a moderate perpendicular field stabilizes a hexagonal lattice of skyrmions in which the spin turns smoothly from aligned with the field on the rim to opposed at the core. Wrapping the unit sphere once around each core gives every skyrmion an integer topological charge that smooth deformations cannot unwind, and a spin-polarized current pushes the lattice sideways at the skyrmion Hall angle.

Each spin is modelled as an intrinsic patch in its plane of precession and each magnetic field as an extrinsic turning plane, so the Zeeman energy pairs them directly as `-(applied_field & spins)`. Chiral exchange transports a neighbour's spin across each bond by a rotor generated from `twist * bonds`, where `twist` is an intrinsic scalar carrying the crystal's handedness, before mapping that transported spin to its exchange field via `exchange(carriers >> tail_spins)` and pairing with `head_spins`. Parallel transport around each triangle yields a non-trivial rotor holonomy whose curvature sets the skyrmion scale, while parallel transport by the shortest-arc spin rotors `(1 + head_spins * tail_spins).normalized()` yields the solid-angle topological charge on every face. Stepping turns each spin by a rotor built from its effective field, current advection, and the Gilbert damping quarter turn `undamped_field.commutator(spins.dual())`, keeping every spin normalized by construction.

In Gibbs–Heaviside vector notation, spins and fields are both written as 3-vectors $\mathbf m_i$ and $\mathbf H_i$, splitting the single rotor-transported exchange into three separate dot- and cross-product sums $E = -\sum_{\langle ij\rangle}\bigl[\cos\theta\,\mathbf m_i\cdot\mathbf m_j + \sin\theta\,\hat{\mathbf r}_{ij}\cdot(\mathbf m_i\times\mathbf m_j) + (1-\cos\theta)(\hat{\mathbf r}_{ij}\cdot\mathbf m_i)(\hat{\mathbf r}_{ij}\cdot\mathbf m_j)\bigr] - h\sum_i m_{z,i}$ with Dzyaloshinskii–Moriya axial vectors $\mathbf D_{ij} = \sin\theta\,\hat{\mathbf r}_{ij}$. Precession, Gilbert damping, current advection, and discrete topological charge then require repeated cross products and triple products in $\dot{\mathbf m} = -\frac{1}{1+\alpha^2}\bigl[\mathbf m\times\tilde{\mathbf H} + \alpha\,\mathbf m\times(\mathbf m\times\tilde{\mathbf H})\bigr]$ with $\tilde{\mathbf H} = \mathbf H_{\mathrm{eff}} + \mathbf H_{\mathrm{th}} - \mathbf m\times(\mathbf u\cdot\nabla)\mathbf m$ and $q_{ijk} = \frac{1}{2\pi}\operatorname{atan2}\bigl(\mathbf m_i\cdot(\mathbf m_j\times\mathbf m_k),\,1 + \mathbf m_i\cdot\mathbf m_j + \mathbf m_j\cdot\mathbf m_k + \mathbf m_k\cdot\mathbf m_i\bigr)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from examples.quantum.skyrmions import render, scenarios
from examples.util import shapes
from examples.util.animation import show_states
from numga import Algebra, stack, where
from numga.algebras import VGA3D as ga
from numga.backend.jax import JaxContext
from numga.topology import Topology

jax.config.update("jax_enable_x64", True)

mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Upright = ga.gatype.vector().derive.within(Algebra("x+y+"))                   # Vector+
Spin = ga.gatype.antibivector()                                               # Antibivector-
Field = ga.gatype.bivector()                                                  # Bivector+
Rotor = ga.gatype.rotor()                                                     # Rotor+
# Perpendicular applied fields turn spins in the film plane.
film = mv.xy                                                                  # [] Field+
# Isotropic exchange maps a neighbour's spin patch to the field plane it exerts.
exchange = Spin.dual()                                                        # [] Field+ <- Spin-

## Periodic triangular lattice

The film is modelled on a periodic triangular lattice wrapped into a flat torus via `shapes.flat_torus`, giving every site six equilateral neighbours and eliminating boundary edges. Because bonds crossing the periodic wrap connect sites on opposite sides of the fundamental rhombus, the lattice carries explicit unit bond displacements alongside its site positions.

In [ ]:
n_sites = 48

In [ ]:
topology, coordinates, steps, spans = shapes.flat_torus(n_sites, n_sites)
V, E, F = topology.n_elements
vertices = mv(Upright, coordinates).dual().field()                            # Point-[V]
bonds = mv(Upright, steps).dual().field()                                     # Point-[E]
periods = mv(Upright, spans).dual()                                           # [2] Point-
render.lattice_figure(topology, vertices, bonds, periods);

Grey segments and dots show the bonds and sites near the origin tiled across neighbouring periodic copies, while the dashed red lines mark the two fundamental periods across which opposite sides wrap.

## Chiral exchange and bond holonomy

Isotropic exchange maps a spin patch to the turning plane it exerts on a neighbour through `exchange = Spin.dual()`, which is the single place where the Euclidean metric enters the Hamiltonian. In a chiral crystal, each bond first transports the tail spin to the head site by a rotor generated from `twist * bonds`, where `twist` is an intrinsic scalar that reverses sign under spatial reflection and sets the handedness of every skyrmion. Expanding that rotor sandwich recovers isotropic Heisenberg exchange weighted by `cos(twist_angle)`, the Dzyaloshinskii–Moriya twist weighted by `sin(twist_angle)`, and a second-order bond anisotropy.

In [ ]:
twist_angle = 0.55

In [ ]:
twist = twist_angle * mv.xyz.dual()                                           # [] Scalar-
twists = twist * bonds                                                        # Field+[E]
carriers = (twists * (-1 / 2)).exp()                                          # Rotor+[E]


def energy(topology: Topology, carriers: Rotor, applied_field: Field, spins: Spin) -> Scalar:
    """Chiral bond exchange across `carriers` (`Rotor+[E]`) and Zeeman pairing with `applied_field` (`[] Field+`) sum to the total energy (`[] Scalar+`) of `spins` (`Spin-[V]`)."""
    tail_spins, head_spins = topology.incidence[1, 0] * spins                 # Spin-[E], Spin-[E]
    exchange_fields = exchange(carriers >> tail_spins)                        # Field+[E]
    return -(exchange_fields & head_spins).sites.sum() - (applied_field & spins).sites.sum()

Composing the three edge rotors around a triangle yields its holonomy, which transports a spin around the boundary and back to its starting corner. When a bond connection has trivial holonomy on every face, gauge-rotating the spins removes the twist completely and reduces the system to a standard ferromagnet, whereas here the non-commuting bond axes curve the connection in the film plane and frustrate uniform alignment on a length scale inversely proportional to `twist_angle`.

In [ ]:
def holonomies(topology: Topology, rotors: Rotor) -> Rotor:
    """Parallel transport around each triangle composes the oriented edge `rotors` (`Rotor+[E]`) into a face holonomy (`Rotor+[F]`)."""
    side_rotors = topology.incidence[2, 1] * rotors                           # [3] Rotor+[F]
    edge_signs = topology.boundary[2, 1] * rotors.norm()                      # [3] Scalar+[F]
    oriented_sides = where(edge_signs > 0, side_rotors, ~side_rotors)         # [3] Rotor+[F]
    # Left-multiplying around the boundary composes each edge transport onto the arriving spin.
    return oriented_sides.cumprod(axis=0)[-1]


curvatures = holonomies(topology, carriers).log().batch()                     # [F] Field+
print("upward triangle:\n", curvatures[0].coefficient_table())
print("downward triangle:\n", curvatures[n_sites ** 2].coefficient_table())

The dominant logarithm component lies in the `film` plane with the same value on both upward and downward triangles, proportional to the triangle area times the square of `twist_angle`, while the smaller out-of-plane components arise at third order from the base corner where the loop begins.

## Effective field and damped spin dynamics

Because the total energy is bilinear in the spins, differentiating with respect to each site's spin collects the exchange fields transported across every incident bond, `carriers >> tail_spins` toward heads and `carriers << head_spins` back toward tails, together with `applied_field`. Under a spin-polarized current, conduction electrons advect the spin texture along `drift`, evaluated on each triangle via its corner simplex gradients and averaged onto vertex dual areas. The term ***simplex gradient*** here means the plane on a triangle that pairs to zero along a facing edge and to one with the step from that edge to its opposite corner. Each spin precesses in its total field while Gilbert damping adds the quarter-turn commutator `undamped_field.commutator(spins.dual())` divided by `1 + damping ** 2`, and exponentiating that turning bivector updates every spin by a unit rotor.

In [ ]:
damping, time_step, n_substeps = 0.5, 0.2, 12
frame_duration_ms = 40

In [ ]:
def effective_field(topology: Topology, carriers: Rotor, applied_field: Field, spins: Spin) -> Field:
    """Parallel transport of neighbour `spins` (`Spin-[V]`) across `carriers` (`Rotor+[E]`) combines with `applied_field` (`[] Field+`) into the site effective field (`Field+[V]`)."""
    tail_spins, head_spins = topology.incidence[1, 0] * spins                 # Spin-[E], Spin-[E]
    # Reversing the rotor sandwich transports a head spin back across its bond to the tail site.
    tail_fields = exchange(carriers << head_spins)                            # Field+[E]
    head_fields = exchange(carriers >> tail_spins)                            # Field+[E]
    return (topology.incidence[1, 0].adjugate() * stack((tail_fields, head_fields))).sum(axis=0) + applied_field

In [ ]:
def dynamics(
    topology: Topology,
    bonds: Point,
    carriers: Rotor,
    spins: Spin,
    applied_fields: Field,
    drifts: Point,
    time_steps: np.ndarray,
    damping: float,
    temperature: float,
    n_substeps: int,
    seed: int,
) -> Iterator[Spin]:
    """Yields the lattice `spins` (`Spin-[V]`) across frames, advancing `n_substeps` damped rotor steps per frame under `applied_fields`, current `drifts`, and thermal kicks."""
    V, E, F = topology.n_elements
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    sides = topology.boundary[2, 1] * bonds                                   # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                       # Spin-[F]
    gradients = (sides | patches.inverse()) / 2                               # [3] Plane+[F]
    areas = patches.norm()                                                    # Scalar+[F]
    dual_areas = (corners.adjugate() * (areas / 3)).sum(axis=0)               # Scalar+[V]

    def substep(spins: Spin, thermal_noise: jax.Array, applied_field: Field, drift: Point, time_step: float) -> Spin:
        """One damped rotor step of duration `time_step` updates `spins` (`Spin-[V]`) under its effective field, thermal kick, and drift advection."""
        field = effective_field(topology, carriers, applied_field, spins)     # Field+[V]
        # Fluctuation-dissipation sets the thermal field spread that holds damped spins at the bath temperature.
        thermal_scale = (2 * damping * temperature / time_step) ** 0.5
        thermal_field = thermal_scale * mv(Field, thermal_noise).field()      # Field+[V]
        # Simplex gradients evaluate the directional derivative along the drift on each triangle before dual-area averaging.
        face_derivatives = ((gradients & drift) * (corners * spins)).sum(axis=0)   # Spin-[F]
        drift_derivatives = (corners.adjugate() * (areas / 3 * face_derivatives)).sum(axis=0) / dual_areas   # Spin-[V]
        undamped_field = field + thermal_field - (spins ^ drift_derivatives)  # Field+[V]
        # Gilbert damping adds a quarter turn about each spin, turning the spin toward its field.
        turning_rate = (undamped_field + damping * undamped_field.commutator(spins.dual())) / (1 + damping ** 2)   # Field+[V]
        step_rotors = (turning_rate * (-time_step / 2)).exp()                 # Rotor+[V]
        return step_rotors >> spins

    @jax.jit
    def frame_step(spins: Spin, key: jax.Array, applied_field: Field, drift: Point, time_step: float) -> Spin:
        """A compiled scan of `n_substeps` rotor steps advances `spins` (`Spin-[V]`) across one frame."""
        noise_samples = jax.random.normal(key, (n_substeps, V, 3))
        return jax.lax.scan(
            lambda spins, sample: (substep(spins, sample, applied_field, drift, time_step), None),
            spins,
            noise_samples,
        )[0]

    n_frames = len(time_steps)
    frame_keys = jax.random.split(jax.random.PRNGKey(seed), n_frames)
    for frame in range(n_frames):
        yield spins
        spins = frame_step(spins, frame_keys[frame], applied_fields[frame], drifts[frame], time_steps[frame])

## Crystallization from random spins

Starting from smoothed random spins at zero temperature and zero current in a moderate perpendicular field, rapid local ordering gives way to slow repulsion between skyrmion cores. Growing the step size geometrically across frames resolves both the initial domain coalescence and the final hexagonal ordering at an even visual pace. Running and rendering the crystallization below takes roughly ten seconds.

In [ ]:
crystal_field, n_crystal_frames, first_time_step = 0.4, 140, 0.014
n_smoothing, seed = 1, 2

In [ ]:
initial_noise = scenarios.smoothed_noise(topology, n_smoothing, seed)         # Spin-[V]
zero_drifts = mv.x.dual() * np.zeros(n_crystal_frames)                        # [n_crystal_frames] Point-
crystal_time_steps = np.geomspace(first_time_step, time_step, n_crystal_frames)   # [n_crystal_frames]
crystal_states = show_states(
    dynamics(topology, bonds, carriers, initial_noise, film * np.full(n_crystal_frames, crystal_field), zero_drifts, crystal_time_steps, damping, 0.0, n_substeps, seed),
    render.spin_frames,
    topology,
    vertices,
    bonds,
    periods,
    name="skyrmions_crystallizing",
    duration_ms=frame_duration_ms,
    colors=render.n_animation_colours,
)

Warm red regions align with the perpendicular field, cool blue cores oppose it, neutral grey bands mark spins lying in the film, and dark flow streaks trace the in-plane spin orientation. Early worm-like reversed domains pinch off into circular skyrmions whose in-plane streaks circulate with identical chirality around every blue core before mutual repulsion settles them into a hexagonal crystal.

## Topological charge from spin holonomy

Each triangle's topological charge measures the fraction of the unit sphere enclosed by its three corner spins. Along every bond, `(1 + head_spins * tail_spins).normalized()` forms the shortest-arc rotor carrying the tail spin into the head spin, defining the spin texture's own parallel-transport connection. Composing those three rotors around a triangle returns the corner spin rotated in its own plane by the spherical excess of the geodesic triangle, so summing over the boundaryless torus gives an exact negative integer equal to minus the number of skyrmions.

In [ ]:
def charges(topology: Topology, spins: Spin) -> Scalar:
    """Spherical solid angles enclosed by each triangle's three `spins` (`Spin-[V]`) give the topological charge per face (`Scalar+[F]`)."""
    tail_spins, head_spins = topology.incidence[1, 0] * spins                 # Spin-[E], Spin-[E]
    # Half the turn of the spin product transports each edge's tail spin along the shortest arc to its head spin.
    shortest_turns = (1 + head_spins * tail_spins).normalized()               # Rotor+[E]
    # The face holonomy logarithm lies in the second corner spin's plane at half the enclosed solid angle.
    half_turns = holonomies(topology, shortest_turns).log()                   # Field+[F]
    corner_spins = topology.incidence[2, 0][1] * spins                        # Spin-[F]
    return -2 * (half_turns & corner_spins) / (4 * np.pi)


crystal = crystal_states[-1]                                                  # Spin-[V]
print(f"crystal topological charge: {charges(topology, crystal).sites.sum().to_array():.6f}")

## Spiral, skyrmion crystal, and ferromagnet regimes

Relaxing the same smoothed initial spins at zero field, moderate field, and strong field side by side exhibits all three equilibrium phases of a bulk chiral magnet. Running and rendering the three-panel comparison below takes roughly twenty seconds.

In [ ]:
regime_fields = np.array([0.0, crystal_field, 1.0])

In [ ]:
regime_runs = (
    dynamics(topology, bonds, carriers, initial_noise, film * np.full(n_crystal_frames, field_strength), zero_drifts, crystal_time_steps, damping, 0.0, n_substeps, seed)
    for field_strength in regime_fields
)
show_states(
    zip(*regime_runs),
    render.comparison_frames,
    topology,
    vertices,
    bonds,
    periods,
    name="skyrmions_regimes",
    duration_ms=frame_duration_ms,
    colors=render.n_animation_colours,
);

In the left panel at zero field, reversed domains elongate into alternating helical stripes whose pitch matches the chiral twist. In the centre panel at moderate field, the same domains pinch into the hexagonal skyrmion crystal, whereas in the right panel at strong field every reversed domain collapses rapidly into the uniform red ferromagnet.

## Field sweep and collapse

Sweeping the perpendicular field downward, upward past the collapse threshold, and back down to zero tests the hysteretic stability of the skyrmion crystal. Weak thermal kicks break the unstable equilibrium of the uniform ferromagnet once the external field vanishes. Running and rendering the sweep below takes roughly ten seconds.

In [ ]:
sweep_turning_frames = scenarios.sweep_turning_frames
n_sweep_frames = scenarios.n_sweep_frames
sweep_schedule = scenarios.sweep_schedule                                     # [n_sweep_frames]
sweep_temperature = scenarios.sweep_temperature

In [ ]:
sweep_drifts = mv.x.dual() * np.zeros(n_sweep_frames)                         # [n_sweep_frames] Point-
sweep_states = show_states(
    dynamics(topology, bonds, carriers, crystal, film * sweep_schedule, sweep_drifts, np.full(n_sweep_frames, time_step), damping, sweep_temperature, n_substeps, seed),
    render.spin_frames,
    topology,
    vertices,
    bonds,
    periods,
    name="skyrmions_sweep",
    duration_ms=frame_duration_ms,
    colors=render.n_animation_colours,
)

Lowering the field first inflates the blue skyrmion cores, and raising the field compresses them until, once only a few lattice spacings wide, each core flips across a single triangle and sheds its topological charge into the uniform ferromagnet. When the field returns to zero, thermal fluctuations nucleate labyrinthine helical stripes with zero net charge rather than restoring isolated skyrmions.

## Current-driven skyrmion Hall drift

Driving a rightward spin-polarized current through the settled crystal advects the spin texture while Gilbert damping drags against the motion, deflecting the topological whirls transversely at the skyrmion Hall angle. Running and rendering the driven crystal below takes roughly ten seconds.

In [ ]:
current_direction = scenarios.current_direction                               # [] Point-
n_drive_frames = scenarios.n_drive_frames
drive_speeds = scenarios.drive_speeds                                         # [n_drive_frames]

In [ ]:
drive_states = show_states(
    dynamics(topology, bonds, carriers, crystal, film * np.full(n_drive_frames, crystal_field), current_direction * drive_speeds, np.full(n_drive_frames, time_step), damping, 0.0, n_substeps, seed),
    render.spin_frames,
    topology,
    vertices,
    bonds,
    periods,
    name="skyrmions_driven",
    duration_ms=frame_duration_ms,
    colors=render.n_animation_colours,
)

After the brief initial hold, the entire hexagonal lattice translates coherently rightward and downward at a steep angle to the horizontal current, keeping every skyrmion circular and preserving the total topological charge.

In [ ]:
# checks
# Cold, undriven Gilbert damping decreases the total energy monotonically across crystallization frames.
crystal_energies = stack([
    energy(topology, carriers, crystal_field * film, spins) for spins in crystal_states
])                                                                            # [n_crystal_frames] Scalar+
assert ((crystal_energies[1:] - crystal_energies[:-1]) <= 1e-10 * crystal_energies[0].abs()).all()
# Rotor updates preserve unit spin norm across every run.
for final_spins in (sweep_states[-1], drive_states[-1]):
    assert ((final_spins.norm().batch() - 1).abs() < 1e-9).all()
# Total topological charge equals minus the count of reversed-spin skyrmion cores.
reversed_faces = (film & (topology.incidence[2, 0].mean(axis=0) * crystal)) < 0
n_skyrmions = topology.subset(np.asarray(reversed_faces), 2).n_components
assert (charges(topology, crystal).sites.sum() + n_skyrmions).abs() < 1e-7
# Above the collapse field, every skyrmion has unwound into the uniform ferromagnet with zero charge.
plateau_states = sweep_states[sweep_turning_frames[2]:sweep_turning_frames[3]]
plateau_charges = stack([charges(topology, spins).sites.sum() for spins in plateau_states])   # [...] Scalar+
assert (plateau_charges.abs() < 1e-7).all()
# Rotor-transported bond exchange equals Heisenberg exchange plus Dzyaloshinskii-Moriya twist plus second-order bond anisotropy.
tail_spins, head_spins = topology.incidence[1, 0] * crystal                   # Spin-[E], Spin-[E]
twist_planes = twists.normalized()                                            # Field+[E]
heisenberg = -np.cos(twist_angle) * (exchange(tail_spins) & head_spins).sites.sum()   # [] Scalar+
dzyaloshinskii_moriya = np.sin(twist_angle) * ((head_spins ^ tail_spins) & twist_planes.dual()).sites.sum()   # [] Scalar+
anisotropy = -(1 - np.cos(twist_angle)) * ((twist_planes & tail_spins) * (twist_planes & head_spins)).sites.sum()   # [] Scalar+
zeeman = -(crystal_field * film & crystal).sites.sum()                        # [] Scalar+
total_energy = energy(topology, carriers, crystal_field * film, crystal)      # [] Scalar+
decomposed_energy = heisenberg + dzyaloshinskii_moriya + anisotropy + zeeman  # [] Scalar+
assert (total_energy - decomposed_energy).abs() < 1e-9 * total_energy.abs()